# Separation: SAM Audio against Demucs

Nov 1. Which separator should serve which sound?

**Data:** AlbumDB (CC BY 4.0), a ten-song album with stems. **Measure:** note agreement with the clean stem, and SI-SDR gain. Metrics and decision rules were written down before results.

**What it decided:** eight draws ranked by Judge lift SAM Audio from 0.281 to 0.479, and Demucs still leads at 0.547. Vocals, bass and guitar stay on SAM Audio by project choice (0.685 against 0.710, n=7, a subgroup chosen after seeing results). Demucs variants and running Demucs before SAM Audio were dropped.

This is a selection of the cells from that work, with the output they printed. They rely on earlier cells and on a Drive folder of datasets and outputs that is not in this repository.


## Ten unseen targets from five songs


In [ ]:
import shutil, subprocess, sys
print("python", sys.version.split()[0])
print("free disk", round(shutil.disk_usage("/content").free / 2**30, 1), "GiB")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
          or "no GPU attached")
except FileNotFoundError:
    print("no GPU attached")
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
BASE = Path("/content/drive/MyDrive/mixray-quality")
for name in ("datasets/albumdb/stems_mixed", "runs"):
    target = BASE / name
    print(name, "->", "ok" if target.is_dir() else "MISSING",
          sorted(p.name for p in target.iterdir())[:12] if target.is_dir() else "")


python 3.13.18
free disk 225.7 GiB
no GPU attached
Mounted at /content/drive
datasets/albumdb/stems_mixed -> ok ['01 Extraordinary', '02 No More Mr Nice Guy', '03 Karma', '04 Magistrates', '05 Diva', '06 Let The Light In', '07 Take On The Seas', '08 An Evening To Myself', '09 Great British Summer', "10 It Ain't Easy (To Mend A Heart)"]
runs -> ok ['dev-002', 'dev-003', 'smoke-001']

python 3.13.16
free disk 235.6 GiB
NVIDIA A100-SXM4-40GB, 40960 MiB
Mounted at /content/drive
datasets/albumdb/stems_mixed -> ok ['01 Extraordinary', '02 No More Mr Nice Guy', '03 Karma', '04 Magistrates', '05 Diva', '06 Let The Light In', '07 Take On The Seas', '08 An Evening To Myself', '09 Great British Summer', "10 It Ain't Easy (To Mend A Heart)"]
runs -> ok ['dev-002', 'dev-003', 'smoke-001']


In [ ]:
"""One-shot setup: permissions, uv, three isolated environments, import probes.

Reuses the exact pins and lockfiles proven in the smoke-001 run, so nothing is re-resolved.
Runs in the background and writes /content/setup.log; poll that instead of blocking.
"""

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/mixray-quality")
PROVEN = BASE / "runs" / "smoke-001"
WORK = Path("/content/mixray-quality-work")
WORK.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ)
ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENV["HF_HOME"] = "/content/hf"
ENV["HF_HUB_DISABLE_TELEMETRY"] = "1"
ENV["TOKENIZERS_PARALLELISM"] = "false"
if not ENV["HF_TOKEN"]:
    raise RuntimeError("Set the HF_TOKEN Colab secret before running this cell")

UV_VERSION = json.loads((PROVEN / "bootstrap.json").read_text())["uv"]
TORCH_PINS = json.loads((PROVEN / "torch-pins.json").read_text())
uv_target = WORK / "uv-tool"
UV = str(uv_target / "bin" / "uv")
if not Path(UV).exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "--no-deps", "--target", str(uv_target), f"uv=={UV_VERSION}"], check=True)

COMMON = '"numpy==1.26.4", "soundfile==0.14.0", "librosa==0.11.0", "mir-eval==0.8.2"'
SAM_GIT = [
    "sam-audio @ git+https://github.com/facebookresearch/sam-audio.git@bb4c6999d2677c7402360e426afc01ddfad6dce0",
    "dacvae @ git+https://github.com/facebookresearch/dacvae.git@414c20785fc3a28373073ea8ef7a1316eeeaca6e",
    "imagebind @ git+https://github.com/facebookresearch/ImageBind.git@53680b02d7e37b19b124fa37bae4b6c98c38f5be",
    "laion-clap @ git+https://github.com/lematt1991/CLAP.git@b85cb7ecac0af7d064cbf03ebd990e709e959fe7",
    "perception-models @ git+https://github.com/facebookresearch/perception_models@e72b6810b1133e1c879f2cc965d276eb73803f1f",
]
_git = ", ".join(f'"{item}"' for item in SAM_GIT)
_overrides = ", ".join(f'"{item}"' for item in SAM_GIT[1:] + ["protobuf==3.20.3"])
_torch = ", ".join(f'"{name}=={version}"' for name, version in TORCH_PINS.items())
_cuda_indexes = "\n".join(
    f'[[tool.uv.index]]\nname = "{name}-cuda"\nurl = "https://download.pytorch.org/whl/cu130"\n'
    f"explicit = true\n" for name in TORCH_PINS
)
_cuda_sources = "\n".join(f'{name} = {{ index = "{name}-cuda" }}' for name in TORCH_PINS)

PYPROJECTS = {
    "sam": f"""[project]
name = "mixray-quality-sam"
version = "0.1.0"
requires-python = "==3.11.*"
dependencies = [{COMMON}, {_git}, "transformers==4.57.1", "huggingface-hub==0.36.2", "protobuf==3.20.3", "tensorboard==2.20.0", "setuptools==80.9.0", {_torch}]
[tool.uv]
package = false
environments = ["sys_platform == 'linux' and platform_machine == 'x86_64'"]
override-dependencies = [{_overrides}]

{_cuda_indexes}
[tool.uv.sources]
{_cuda_sources}
""",
    "demucs": f"""[project]
name = "mixray-quality-demucs"
version = "0.1.0"
requires-python = "==3.11.*"
dependencies = [{COMMON}, "demucs==4.1.0", "torch==2.15.0"]
[tool.uv]
package = false
environments = ["sys_platform == 'linux' and platform_machine == 'x86_64'"]

[[tool.uv.index]]
name = "cpu"
url = "https://download.pytorch.org/whl/cpu"
explicit = true
[tool.uv.sources]
torch = {{ index = "cpu" }}
""",
    "pitch": f"""[project]
name = "mixray-quality-pitch"
version = "0.1.0"
requires-python = "==3.11.*"
dependencies = [{COMMON}, "basic-pitch==0.4.0", "tensorflow==2.15.0", "setuptools==80.9.0"]
[tool.uv]
package = false
environments = ["sys_platform == 'linux' and platform_machine == 'x86_64'"]
""",
}
PROJECTS = {}
for name, contents in PYPROJECTS.items():
    project = WORK / name
    project.mkdir(parents=True, exist_ok=True)
    (project / "pyproject.toml").write_text(contents)
    # The lockfile from the proven run keeps resolution identical and fast.
    lock = PROVEN / f"{name}-uv.lock"
    if lock.is_file():
        shutil.copy2(lock, project / "uv.lock")
    PROJECTS[name] = project

PROBES = {
    "pitch": "import tensorflow; from basic_pitch.inference import predict; import librosa, soundfile, mir_eval",
    "demucs": "import torch; import demucs.separate; from demucs.pretrained import get_model",
    "sam": ("from tensorboard.compat.proto import event_pb2; import torch, torchaudio, torchvision; "
            "from sam_audio import SAMAudio, SAMAudioProcessor; "
            "from sam_audio.ranking import create_ranker; from torchcodec.decoders import AudioDecoder; "
            "assert torch.cuda.is_available(), 'no GPU'; "
            "x = torch.ones((32, 32), device='cuda'); assert float((x @ x)[0, 0]) == 32"),
}
script = ["set -o pipefail", "echo SETUP_START"]
for name, project in PROJECTS.items():
    locked = "--locked" if (project / "uv.lock").is_file() else ""
    script.append(f'echo "--- sync {name} ---"')
    script.append(f'{UV} sync --project {project} --python 3.11 {locked} || exit 1')
    script.append(f'echo "--- probe {name} ---"')
    script.append(
        f'{UV} run --no-sync --project {project} python -c {json.dumps(PROBES[name])} || exit 1'
    )
script.append("echo SETUP_DONE")
setup_script = WORK / "setup.sh"
setup_script.write_text("\n".join(script))

LOG = Path("/content/setup.log")
LOG.write_text("")
with LOG.open("w") as handle:
    process = subprocess.Popen(["bash", str(setup_script)], env=ENV, stdout=handle,
                               stderr=subprocess.STDOUT, start_new_session=True)
Path("/content/setup.pid").write_text(str(process.pid))
print(f"setup running as pid {process.pid}; poll /content/setup.log")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
setup running as pid 7394; poll /content/setup.log

Mounted at /content/drive
setup running as pid 4912; poll /content/setup.log


In [ ]:
"""Held-out validation: ten cases from the five songs never used in smoke-001/dev-002/dev-003.

Four targets (organ, cello, synthesizer) fall outside Demucs' six categories on purpose: that is
where SAM Audio has to carry the product, and it has never been measured there.
"""

import json
import shutil
import subprocess
from pathlib import Path

RUN_ID = "holdout-004"
MODEL_ID = "facebook/sam-audio-large"
DRAWS = 2
RUN = BASE / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
SONGS = BASE / "datasets" / "albumdb" / "stems_mixed"
WORKER = WORK / "quality_worker.py"
shutil.copy2(PROVEN / "quality_worker.py", WORKER)


def stems(song: str, *names: str) -> list[str]:
    return [str(SONGS / song / name) for name in names]


# Each song contributes one mixture and two targets, so a pair shares one offset and one mix.
GROUPS = {
    "nice_guy": {
        "song": "02 No More Mr Nice Guy",
        "stems": stems("02 No More Mr Nice Guy", "bass.wav", "drums_kick.wav",
                       "drums_overheads.wav", "vox_lead.wav", "guitar_acoustic.wav",
                       "guitar_electric_solo_l.wav"),
        "cases": [("nice_guy_vox", "vocals", "lead vocals", "vox_lead.wav"),
                  ("nice_guy_solo", "guitar", "lead electric guitar",
                   "guitar_electric_solo_l.wav")],
    },
    "karma": {
        "song": "03 Karma",
        "stems": stems("03 Karma", "bass.wav", "drums_kick.wav", "drums_overheads.wav",
                       "vox_lead.wav", "keys_piano.wav", "guitar_acoustic_main.wav"),
        "cases": [("karma_piano", "piano", "piano", "keys_piano.wav"),
                  ("karma_acoustic", "guitar", "acoustic guitar", "guitar_acoustic_main.wav")],
    },
    "evening": {
        "song": "08 An Evening To Myself",
        "stems": stems("08 An Evening To Myself", "bass.wav", "drums_kick.wav",
                       "drums_overheads.wav", "vox_lead.wav", "keys_organ.wav",
                       "guitar_electric_lead.wav"),
        "cases": [("evening_organ", "organ", "organ", "keys_organ.wav"),
                  ("evening_bass", "bass", "bass guitar", "bass.wav")],
    },
    "summer": {
        "song": "09 Great British Summer",
        "stems": stems("09 Great British Summer", "bass.wav", "drums_kick.wav",
                       "drums_overheads.wav", "vox_lead.wav", "guitar_acoustic_solo.wav",
                       "keys_synth_1.wav"),
        "cases": [("summer_acoustic", "guitar", "acoustic guitar", "guitar_acoustic_solo.wav"),
                  ("summer_synth", "synth", "synthesizer", "keys_synth_1.wav")],
    },
    "mend": {
        "song": "10 It Ain't Easy (To Mend A Heart)",
        "stems": stems("10 It Ain't Easy (To Mend A Heart)", "bass.wav", "vox_lead.wav",
                       "guitar_acoustic_l.wav", "guitar_acoustic_solo.wav", "cello.wav",
                       "violin_1.wav"),
        "cases": [("mend_cello", "cello", "cello", "cello.wav"),
                  ("mend_vox", "vocals", "lead vocals", "vox_lead.wav")],
    },
}

missing = sorted({p for g in GROUPS.values() for p in g["stems"] if not Path(p).is_file()})
if missing:
    raise FileNotFoundError("Fix these stem paths before spending GPU time:\n" + "\n".join(missing))

# dev-002 wasted a case because only a group's first target was checked for activity.
ENERGY_SCAN = r'''
import json, sys
import numpy as np
import soundfile as sf

WINDOW = 30
MIN_TARGET_TO_MIX_DB = -20.0
groups = json.loads(sys.stdin.read())
chosen = {}
for name, spec in groups.items():
    curves, rate, seconds = [], None, None
    for path in spec["targets"]:
        audio, sample_rate = sf.read(path, dtype="float64", always_2d=True)
        samples = audio.mean(axis=1)
        whole = len(samples) // sample_rate
        if whole < WINDOW:
            raise ValueError(name + ": target shorter than the clip")
        edges = np.arange(0, whole * sample_rate, sample_rate)
        per_second = np.add.reduceat(samples[: whole * sample_rate] ** 2, edges)
        cumulative = np.concatenate(([0.0], np.cumsum(per_second)))
        totals = cumulative[WINDOW:] - cumulative[:-WINDOW]
        if rate is None:
            rate, seconds = sample_rate, len(totals)
        elif sample_rate != rate:
            raise ValueError(name + ": targets disagree on sample rate")
        seconds = min(seconds, len(totals))
        curves.append(totals)
    shares = np.stack([c[:seconds] / max(c[:seconds].max(), 1e-30) for c in curves])
    start = int(np.argmax(shares.min(axis=0)))
    begin, stop = start * rate, (start + WINDOW) * rate
    mixed = None
    for path in spec["stems"]:
        audio, _ = sf.read(path, dtype="float64", always_2d=True, start=begin, stop=stop)
        samples = audio.mean(axis=1)
        mixed = samples if mixed is None else mixed + samples
    mix_rms = float(np.sqrt(np.mean(mixed**2)))
    levels = {}
    for path, curve in zip(spec["targets"], curves):
        rms = float(np.sqrt(curve[start] / (WINDOW * rate)))
        relative = 20.0 * np.log10(max(rms, 1e-30) / max(mix_rms, 1e-30))
        levels[path] = {"rms": rms, "to_mix_db": float(relative)}
        if relative < MIN_TARGET_TO_MIX_DB:
            raise ValueError(
                f"{name}: {path} sits {relative:.1f} dB under the mixture at its own best window"
            )
    chosen[name] = {"start": float(start), "mix_rms": mix_rms, "levels": levels}
print("ENERGY_SCAN " + json.dumps(chosen))
'''

request = {
    key: {"stems": group["stems"],
          "targets": [str(SONGS / group["song"] / target) for *_, target in group["cases"]]}
    for key, group in GROUPS.items()
}
scan_env = dict(ENV)
scan_env["CUDA_VISIBLE_DEVICES"] = "-1"
completed = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", ENERGY_SCAN],
    input=json.dumps(request), env=scan_env, capture_output=True, text=True,
    check=True, timeout=1800,
)
marker = [line for line in completed.stdout.splitlines() if line.startswith("ENERGY_SCAN ")]
if len(marker) != 1:
    raise RuntimeError("Energy scan produced no offsets:\n" + completed.stdout + completed.stderr)
offsets = json.loads(marker[0].removeprefix("ENERGY_SCAN "))
(RUN / "offsets.json").write_text(json.dumps(offsets, indent=2))

CASES = []
for key, group in GROUPS.items():
    picked = offsets[key]
    for identifier, instrument, prompt, target in group["cases"]:
        path = str(SONGS / group["song"] / target)
        level = picked["levels"][path]["to_mix_db"]
        print(f"  {identifier:18s} start {picked['start']:5.0f} s  {level:+6.1f} dB vs mixture  "
              f"'{prompt}'")
        CASES.append({
            "id": identifier, "song": group["song"], "split": "dev", "instrument": instrument,
            "prompt": prompt, "source_url": "https://zenodo.org/records/19683001",
            "license": "CC-BY-4.0", "stems": group["stems"], "targets": [path],
            "start": picked["start"], "duration": 30.0, "notes": None,
        })

CONFIG = {"output": str(RUN), "model": MODEL_ID, "draws": DRAWS,
          "allow_holdout": False, "cases": CASES}
config_path = RUN / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise ValueError("Configuration changed. Choose a new RUN_ID to retain the old baseline")
config_path.write_text(json.dumps(CONFIG, indent=2))

STAGES = [("prepare", "pitch"), ("sam", "sam"), ("demucs", "demucs"),
          ("transcribe", "pitch"), ("report", "pitch")]
lines = ["set -o pipefail", "echo HOLDOUT_START"]
for stage, project in STAGES:
    prefix = "CUDA_VISIBLE_DEVICES=-1 " if project in {"pitch", "demucs"} else ""
    lines.append(f'echo "--- {stage} ---"')
    lines.append(f"{prefix}{UV} run --no-sync --project {PROJECTS[project]} python "
                 f"{WORKER} {stage} {config_path} || exit 1")
lines.append("echo HOLDOUT_DONE")
script = WORK / "holdout.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/holdout.log"
JOB_PID = "/content/holdout.pid"
JOB_DONE_MARKER = "HOLDOUT_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"\n{len(CASES)} cases; pipeline running as pid {job.pid}; poll {JOB_LOG}")


CalledProcessError: Command '['/content/mixray-quality-work/uv-tool/bin/uv', 'run', '--no-sync', '--project', '/content/mixray-quality-work/pitch', 'python', '-c', '\nimport json, sys\nimport numpy as np\nimport soundfile as sf\n\nWINDOW = 30\nMIN_TARGET_TO_MIX_DB = -20.0\ngroups = json.loads(sys.stdin.read())\nchosen = {}\nfor name, spec in groups.items():\n    curves, rate, seconds = [], None, None\n    for path in spec["targets"]:\n        audio, sample_rate = sf.read(path, dtype="float64", always_2d=True)\n        samples = audio.mean(axis=1)\n        whole = len(samples) // sample_rate\n        if whole < WINDOW:\n            raise ValueError(name + ": target shorter than the clip")\n        edges = np.arange(0, whole * sample_rate, sample_rate)\n        per_second = np.add.reduceat(samples[: whole * sample_rate] ** 2, edges)\n        cumulative = np.concatenate(([0.0], np.cumsum(per_second)))\n        totals = cumulative[WINDOW:] - cumulative[:-WINDOW]\n        if rate is None:\n

In [ ]:
probe = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", ENERGY_SCAN],
    input=json.dumps(request), env=scan_env, capture_output=True, text=True, timeout=1800,
)
print("exit", probe.returncode)
print(probe.stdout[-2000:])
print("--- stderr ---")
print(probe.stderr[-3000:])


exit 1

--- stderr ---
Traceback (most recent call last):
  File "<string>", line 43, in <module>
ValueError: evening: /content/drive/MyDrive/mixray-quality/datasets/albumdb/stems_mixed/08 An Evening To Myself/keys_organ.wav sits -20.3 dB under the mixture at its own best window


In [ ]:
"""Continue the held-out run: the -20 dB bar rejected a genuinely quiet organ, not a dead stem.

The guard exists to catch absent targets like dev-002's -98 dB rhythm guitar, so the floor moves to
-30 dB and every level is printed instead. Reuses GROUPS/request from the previous cell.
"""

ENERGY_SCAN = ENERGY_SCAN.replace("MIN_TARGET_TO_MIX_DB = -20.0", "MIN_TARGET_TO_MIX_DB = -30.0")

completed = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", ENERGY_SCAN],
    input=json.dumps(request), env=scan_env, capture_output=True, text=True, timeout=1800,
)
if completed.returncode:
    raise RuntimeError("Energy scan failed:\n" + completed.stdout + completed.stderr)
marker = [line for line in completed.stdout.splitlines() if line.startswith("ENERGY_SCAN ")]
if len(marker) != 1:
    raise RuntimeError("Energy scan produced no offsets:\n" + completed.stdout)
offsets = json.loads(marker[0].removeprefix("ENERGY_SCAN "))
(RUN / "offsets.json").write_text(json.dumps(offsets, indent=2))

CASES = []
for key, group in GROUPS.items():
    picked = offsets[key]
    for identifier, instrument, prompt, target in group["cases"]:
        path = str(SONGS / group["song"] / target)
        level = picked["levels"][path]["to_mix_db"]
        flag = "  <- quiet" if level < -15.0 else ""
        print(f"  {identifier:18s} start {picked['start']:5.0f} s  {level:+6.1f} dB vs mixture  "
              f"'{prompt}'{flag}")
        CASES.append({
            "id": identifier, "song": group["song"], "split": "dev", "instrument": instrument,
            "prompt": prompt, "source_url": "https://zenodo.org/records/19683001",
            "license": "CC-BY-4.0", "stems": group["stems"], "targets": [path],
            "start": picked["start"], "duration": 30.0, "notes": None,
        })

CONFIG = {"output": str(RUN), "model": MODEL_ID, "draws": DRAWS,
          "allow_holdout": False, "cases": CASES}
config_path = RUN / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise ValueError("Configuration changed. Choose a new RUN_ID to retain the old baseline")
config_path.write_text(json.dumps(CONFIG, indent=2))

STAGES = [("prepare", "pitch"), ("sam", "sam"), ("demucs", "demucs"),
          ("transcribe", "pitch"), ("report", "pitch")]
lines = ["set -o pipefail", "echo HOLDOUT_START"]
for stage, project in STAGES:
    prefix = "CUDA_VISIBLE_DEVICES=-1 " if project in {"pitch", "demucs"} else ""
    lines.append(f'echo "--- {stage} ---"')
    lines.append(f"{prefix}{UV} run --no-sync --project {PROJECTS[project]} python "
                 f"{WORKER} {stage} {config_path} || exit 1")
lines.append("echo HOLDOUT_DONE")
script = WORK / "holdout.sh"
script.write_text("\n".join(lines))

JOB_LOG = "/content/holdout.log"
JOB_PID = "/content/holdout.pid"
JOB_DONE_MARKER = "HOLDOUT_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"\n{len(CASES)} cases; pipeline running as pid {job.pid}; poll {JOB_LOG}")


  nice_guy_vox       start   165 s    -5.5 dB vs mixture  'lead vocals'
  nice_guy_solo      start   165 s    -7.2 dB vs mixture  'lead electric guitar'
  karma_piano        start   160 s   -15.3 dB vs mixture  'piano'  <- quiet
  karma_acoustic     start   160 s    -5.0 dB vs mixture  'acoustic guitar'
  evening_organ      start    95 s   -20.3 dB vs mixture  'organ'  <- quiet
  evening_bass       start    95 s    -3.6 dB vs mixture  'bass guitar'
  summer_acoustic    start   150 s    -5.2 dB vs mixture  'acoustic guitar'
  summer_synth       start   150 s   -26.5 dB vs mixture  'synthesizer'  <- quiet
  mend_cello         start   137 s    -6.8 dB vs mixture  'cello'
  mend_vox           start   137 s    -3.9 dB vs mixture  'lead vocals'

10 cases; pipeline running as pid 9726; poll /content/holdout.log


## Frozen metrics, and a selector that cross-checks SAM Audio against Demucs


In [ ]:
"""M2 and M3: frozen metrics over every run, plus the cross-check selector.

Writes /content/analysis.py and runs it in the pitch environment, which holds mir_eval.
"""

ANALYSIS = r'''
"""Score every case under the frozen metrics S1, N1, N2 and test the cross-check selector."""

import json
import sys
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000
ONSET_TOLERANCE = 0.05
PITCH_TOLERANCE = 50.0
DEMUCS_STEMS = ("vocals", "drums", "bass", "guitar", "piano", "other")
# Mirrors engine/fallback.py PROMPT_STEM_KEYWORDS, so the routing tested here is the shipped one.
PROMPT_STEM_KEYWORDS = (
    ("vocals", {"vocal", "vocals", "voice", "sing", "singing", "singer"}),
    ("drums", {"drum", "drums", "kick", "snare", "hihat", "hat", "cymbal"}),
    ("bass", {"bass", "sub", "subbass"}),
    ("guitar", {"guitar", "guitars"}),
    ("piano", {"piano", "keys", "keyboard"}),
)


def stem_for_prompt(prompt):
    tokens = set(prompt.lower().split())
    for stem, keywords in PROMPT_STEM_KEYWORDS:
        if tokens & keywords:
            return stem
    return None


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def si_sdr(reference, estimate):
    shared = min(len(reference), len(estimate))
    target = reference[:shared] - reference[:shared].mean()
    audio = estimate[:shared] - estimate[:shared].mean()
    power = float(target @ target)
    if power < 1e-12:
        return float("nan")
    projection = target * float(audio @ target) / power
    error = audio - projection
    return 10.0 * np.log10(
        (float(projection @ projection) + 1e-12) / (float(error @ error) + 1e-12)
    )


def note_arrays(path):
    notes = sorted(json.loads(Path(path).read_text())["notes"], key=lambda e: e[0])
    intervals = np.array([[a, b] for a, b, _ in notes], dtype=float).reshape(-1, 2)
    pitches = np.array([p for _, _, p in notes], dtype=float)
    return intervals, 440.0 * 2 ** ((pitches - 69) / 12)


def chroma(frequencies):
    return mir_eval.util.midi_to_hz(mir_eval.util.hz_to_midi(frequencies) % 12 + 60)


def f1(reference_intervals, reference_hz, intervals, frequencies):
    if len(reference_hz) == 0 or len(frequencies) == 0:
        return float("nan")
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference_intervals, reference_hz, intervals, frequencies,
        onset_tolerance=ONSET_TOLERANCE, pitch_tolerance=PITCH_TOLERANCE, offset_ratio=None,
    )
    return float(value)


rows = []
for run_dir in sys.argv[1:]:
    run = Path(run_dir)
    config = json.loads((run / "config.json").read_text())
    prompts = {case["id"]: case["prompt"] for case in config["cases"]}
    offsets = {}
    if (run / "offsets.json").is_file():
        for group in json.loads((run / "offsets.json").read_text()).values():
            offsets.update({Path(p).name: v["to_mix_db"] for p, v in group["levels"].items()})
    targets = {case["id"]: Path(case["targets"][0]).name for case in config["cases"]}
    for case_dir in sorted(p for p in run.iterdir() if p.is_dir()):
        case = case_dir.name
        artifact = case_dir / "sam-audio-large"
        folder = artifact / "transcription"
        if not (folder / "reference.json").is_file():
            continue
        prompt = prompts.get(case, "?")
        requested = stem_for_prompt(prompt)
        reference = mono(case_dir / "reference.wav")
        length = len(reference)
        mixture = mono(case_dir / "mix.wav", length)
        stems = {}
        for stem in DEMUCS_STEMS:
            path = case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{stem}.wav"
            if path.is_file():
                stems[stem] = mono(path, length)
        reference_intervals, reference_hz = note_arrays(folder / "reference.json")
        mix_separation = si_sdr(reference, mixture)

        candidates = {}
        for path in sorted(artifact.glob("candidate-*.wav")):
            drawn = mono(path, length)
            against = {stem: si_sdr(samples, drawn) for stem, samples in stems.items()}
            closest = max(against, key=against.get) if against else None
            transcript = folder / f"{path.stem}.json"
            intervals, frequencies = note_arrays(transcript)
            candidates[path.stem] = {
                "separation": si_sdr(reference, drawn) - mix_separation,
                "note_f1": f1(reference_intervals, reference_hz, intervals, frequencies),
                "chroma_f1": f1(reference_intervals, chroma(reference_hz), intervals,
                                chroma(frequencies)),
                "closest_stem": closest,
                "to_requested": against.get(requested) if requested else None,
            }
        judged = None
        if (artifact / "judge.json").is_file():
            judged = json.loads((artifact / "judge.json").read_text()).get("selected")
        demucs_row = None
        if (folder / "demucs.json").is_file() and requested in stems:
            intervals, frequencies = note_arrays(folder / "demucs.json")
            demucs_row = {
                "separation": si_sdr(reference, stems[requested]) - mix_separation,
                "note_f1": f1(reference_intervals, reference_hz, intervals, frequencies),
                "chroma_f1": f1(reference_intervals, chroma(reference_hz), intervals,
                                chroma(frequencies)),
            }
        mix_intervals, mix_hz = note_arrays(folder / "mix.json")
        rows.append({
            "run": run.name, "case": case, "prompt": prompt, "requested_stem": requested,
            "target_to_mix_db": offsets.get(targets.get(case, ""), None),
            "mix": {"note_f1": f1(reference_intervals, reference_hz, mix_intervals, mix_hz),
                    "chroma_f1": f1(reference_intervals, chroma(reference_hz), mix_intervals,
                                    chroma(mix_hz))},
            "candidates": candidates, "judge_selected": judged, "demucs": demucs_row,
        })

Path("/content/analysis.json").write_text(json.dumps(rows, indent=2))
print(f"ANALYSIS_ROWS {len(rows)}")
'''

Path("/content/analysis.py").write_text(ANALYSIS)
runs = [str(BASE / "runs" / name) for name in ("dev-002", "dev-003", "holdout-004")]
analysis_env = dict(ENV)
analysis_env["CUDA_VISIBLE_DEVICES"] = "-1"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/analysis.py", *runs],
    env=analysis_env, capture_output=True, text=True, timeout=3600,
)
print("exit", done.returncode)
print(done.stdout[-1500:])
if done.returncode:
    print("--- stderr ---")
    print(done.stderr[-3000:])


exit 0
ANALYSIS_ROWS 20


In [ ]:
"""Render M2 and M3 against the frozen metrics and the decision rules."""

import json
from statistics import mean

rows = json.loads(Path("/content/analysis.json").read_text())


def finite(values):
    return [v for v in values if v is not None and v == v]


def selected(row):
    """The candidate Judge picked, which is what production serves today."""
    name = row["judge_selected"]
    if name in row["candidates"]:
        return row["candidates"][name]
    return max(row["candidates"].values(), key=lambda c: c["note_f1"])


print("N1 = note F1 vs Basic Pitch on the clean stem (agreement, not ground truth)\n")
header = (f"{'case':20s} {'prompt':22s} {'dB':>6s} {'stem':>7s} {'mix':>6s} "
          f"{'SAM':>6s} {'Demucs':>7s} {'SAM closest':>12s}")
print(header)
print("-" * len(header))
for row in rows:
    sam = selected(row)
    demucs = row["demucs"]
    level = row["target_to_mix_db"]
    level_text = f"{level:+.1f}" if level is not None else "?"
    demucs_text = f"{demucs['note_f1']:.3f}" if demucs else "-"
    print(f"{row['case']:20s} {row['prompt']:22s} {level_text:>6s} "
          f"{(row['requested_stem'] or '-'):>7s} "
          f"{row['mix']['note_f1']:6.3f} {sam['note_f1']:6.3f} {demucs_text:>7s} "
          f"{(sam['closest_stem'] or '-'):>12s}")

covered = [r for r in rows if r["requested_stem"] and r["demucs"]]
uncovered = [r for r in rows if not r["requested_stem"]]
print(f"\n{'-' * 70}")
print(f"Demucs covers the prompt ({len(covered)} cases)")
print(f"  mix     N1 {mean(finite(r['mix']['note_f1'] for r in covered)):.3f}")
print(f"  SAM     N1 {mean(finite(selected(r)['note_f1'] for r in covered)):.3f}")
print(f"  Demucs  N1 {mean(finite(r['demucs']['note_f1'] for r in covered)):.3f}")
print(f"\nPrompt outside Demucs' six categories ({len(uncovered)} cases)")
for row in uncovered:
    print(f"  {row['case']:20s} {row['prompt']:14s} mix {row['mix']['note_f1']:.3f}  "
          f"SAM {selected(row)['note_f1']:.3f}")
if uncovered:
    print(f"  mean{'':17s}{'':14s} mix {mean(finite(r['mix']['note_f1'] for r in uncovered)):.3f}  "
          f"SAM {mean(finite(selected(r)['note_f1'] for r in uncovered)):.3f}")

print(f"\n{'-' * 70}\nM3 cross-check: does SAM's output resemble the stem that was asked for?")
agree_count = 0
simulated, demucs_only = [], []
for row in covered:
    sam, demucs = selected(row), row["demucs"]
    agrees = sam["closest_stem"] == row["requested_stem"]
    better = sam["note_f1"] >= demucs["note_f1"]
    agree_count += int(agrees == better)
    simulated.append(sam["note_f1"] if agrees else demucs["note_f1"])
    demucs_only.append(demucs["note_f1"])
    if agrees != better:
        print(f"  disagrees: {row['case']:20s} cross-check "
              f"{'pass' if agrees else 'fail'}, SAM {sam['note_f1']:.3f} vs "
              f"Demucs {demucs['note_f1']:.3f}")
print(f"  rule predicts which separator wins: {agree_count}/{len(covered)}")
print(f"  Demucs alone          N1 {mean(finite(demucs_only)):.3f}")
print(f"  cross-check selector  N1 {mean(finite(simulated)):.3f}")
regressions = [row["case"] for row, value, baseline in zip(covered, simulated, demucs_only)
               if value < baseline - 1e-9]
print(f"  cases the selector makes worse than Demucs alone: {regressions or 'none'}")


N1 = note F1 vs Basic Pitch on the clean stem (agreement, not ground truth)

case                 prompt                     dB    stem    mix    SAM  Demucs  SAM closest
---------------------------------------------------------------------------------------------
diva_synth           synthesizer                 ?       -  0.000  0.000       -         bass
ex_bass              bass guitar                 ?    bass  0.449  0.812   0.765         bass
ex_guitar_lead       lead electric guitar        ?  guitar  0.435  0.799   0.804       guitar
ex_guitar_rhythm     rhythm electric guitar      ?  guitar  0.000  0.000   0.000         bass
mag_piano_full       piano                       ?   piano  0.024  0.000   0.023         bass
mag_piano_sparse     piano                       ?   piano  0.008  0.000   0.070         bass
light_guitar_rhythm  rhythm electric guitar  -14.5  guitar  0.120  0.027   0.585         bass
light_guitar_solo    lead electric guitar     -8.3  guitar  0.196  0.000   0.

## Latency, memory and cost


In [ ]:
"""M5: latency, peak VRAM and cost for the configuration we ship.

Mirrors production exactly: sam-audio-large with the rankers dropped, the Judge ranker resident,
one draw, and the mix ranked as a second candidate. Reports whether a 24 GB GPU would hold it.
"""

PERF = r'''
"""Measure load, draw and ranking separately so a resident worker can be costed properly."""

import json
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf

RATE = 48000
RANKERS_OFF = {"visual_ranker": None, "text_ranker": None, "span_predictor": None}
MODAL_A100_40GB_PER_SECOND = 0.000583
L4_LIMIT_GIB = 24.0


def gib(value):
    return value / 2**30


def main():
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor
    from sam_audio.model.config import JudgeRankerConfig
    from sam_audio.ranking import create_ranker

    case_dir, model_id, prompt = Path(sys.argv[1]), sys.argv[2], sys.argv[3]
    revision = json.loads((case_dir / model_id.split("/")[-1] / "model.json").read_text())["revision"]
    snapshot = snapshot_download(model_id, revision=revision,
                                 allow_patterns=["*.json", "*.pt", "*.safetensors"])
    results = {"gpu": torch.cuda.get_device_name(0),
               "total_gib": round(gib(torch.cuda.get_device_properties(0).total_memory), 2)}

    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()
    model = SAMAudio.from_pretrained(snapshot, **RANKERS_OFF).eval().cuda()
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    results["model_load_seconds"] = round(time.monotonic() - started, 1)
    results["model_resident_gib"] = round(gib(torch.cuda.memory_allocated()), 2)

    started = time.monotonic()
    ranker = create_ranker(JudgeRankerConfig())
    if ranker is None:
        raise RuntimeError("Judge ranker unavailable")
    ranker = ranker.eval().cuda()
    results["judge_load_seconds"] = round(time.monotonic() - started, 1)
    results["both_resident_gib"] = round(gib(torch.cuda.memory_allocated()), 2)

    audio, rate = sf.read(str(case_dir / "mix.wav"), dtype="float64", always_2d=True)
    if rate != RATE:
        raise RuntimeError("clip is not 48 kHz")
    mixture = np.asarray(audio.mean(axis=1), dtype=np.float32)
    batch = processor(audios=[torch.tensor(mixture).unsqueeze(0)],
                      descriptions=[prompt]).to("cuda")

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.monotonic()
    with torch.inference_mode():
        separation = model.separate(batch, predict_spans=False, reranking_candidates=1)
    torch.cuda.synchronize()
    results["draw_seconds"] = round(time.monotonic() - started, 1)
    results["peak_during_draw_gib"] = round(gib(torch.cuda.max_memory_allocated()), 2)
    target = separation.target[0] if isinstance(separation.target, list) else separation.target
    drawn = target.detach().float().reshape(-1)[: len(mixture)]

    mix_tensor = torch.tensor(mixture, device="cuda")[: len(drawn)]
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.monotonic()
    with torch.inference_mode():
        scores = ranker(input_audio=[mix_tensor.unsqueeze(0).expand(2, -1)],
                        extracted_audio=[torch.stack([drawn, mix_tensor])],
                        descriptions=[prompt], sample_rate=RATE)
    torch.cuda.synchronize()
    results["rank_seconds"] = round(time.monotonic() - started, 1)
    results["peak_during_rank_gib"] = round(gib(torch.cuda.max_memory_allocated()), 2)
    results["scores"] = [round(float(v), 3) for v in scores[0].detach().float().cpu().tolist()]

    gpu_seconds = results["draw_seconds"] + results["rank_seconds"]
    results["gpu_seconds_per_isolation"] = round(gpu_seconds, 1)
    results["dollars_per_isolation_a100_40gb"] = round(
        gpu_seconds * MODAL_A100_40GB_PER_SECOND, 4
    )
    results["peak_overall_gib"] = max(results["peak_during_draw_gib"],
                                      results["peak_during_rank_gib"])
    results["fits_24_gib_gpu"] = results["peak_overall_gib"] < L4_LIMIT_GIB
    print("PERF " + json.dumps(results))


if __name__ == "__main__":
    main()
'''

Path("/content/perf.py").write_text(PERF)
perf_case = BASE / "runs" / "holdout-004" / "mend_cello"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python",
     "/content/perf.py", str(perf_case), MODEL_ID, "cello"],
    env=ENV, capture_output=True, text=True, timeout=1800,
)
line = [l for l in done.stdout.splitlines() if l.startswith("PERF ")]
if not line:
    print("exit", done.returncode)
    print(done.stdout[-1500:])
    print("--- stderr ---")
    print(done.stderr[-2500:])
else:
    perf = json.loads(line[0].removeprefix("PERF "))
    (BASE / "runs" / "holdout-004" / "performance.json").write_text(json.dumps(perf, indent=2))
    for key, value in perf.items():
        print(f"  {key:36s} {value}")


  gpu                                  NVIDIA A100-SXM4-40GB
  total_gib                            39.49
  model_load_seconds                   40.7
  model_resident_gib                   14.32
  judge_load_seconds                   21.0
  both_resident_gib                    20.05
  draw_seconds                         10.9
  peak_during_draw_gib                 28.15
  rank_seconds                         5.8
  peak_during_rank_gib                 28.38
  scores                               [1.649, 1.135]
  gpu_seconds_per_isolation            16.7
  dollars_per_isolation_a100_40gb      0.0097
  peak_overall_gib                     28.38
  fits_24_gib_gpu                      False


In [ ]:
"""M5b: the same clip with Judge removed, to cost the ranker rather than infer it by subtraction."""

PERF_NO_JUDGE = r'''
"""One draw, no ranker resident: measures what dropping Judge actually buys."""

import json
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf

RATE = 48000
RANKERS_OFF = {"visual_ranker": None, "text_ranker": None, "span_predictor": None}
MODAL_RATES_PER_SECOND = {"A100-40GB": 0.000583, "L4": 0.000222}
L4_LIMIT_GIB = 24.0


def gib(value):
    return value / 2**30


def main():
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor

    case_dir, model_id, prompt = Path(sys.argv[1]), sys.argv[2], sys.argv[3]
    revision = json.loads(
        (case_dir / model_id.split("/")[-1] / "model.json").read_text()
    )["revision"]
    snapshot = snapshot_download(model_id, revision=revision,
                                 allow_patterns=["*.json", "*.pt", "*.safetensors"])
    results = {}
    started = time.monotonic()
    model = SAMAudio.from_pretrained(snapshot, **RANKERS_OFF).eval().cuda()
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    results["model_load_seconds"] = round(time.monotonic() - started, 1)
    results["model_resident_gib"] = round(gib(torch.cuda.memory_allocated()), 2)

    audio, rate = sf.read(str(case_dir / "mix.wav"), dtype="float64", always_2d=True)
    if rate != RATE:
        raise RuntimeError("clip is not 48 kHz")
    mixture = np.asarray(audio.mean(axis=1), dtype=np.float32)
    batch = processor(audios=[torch.tensor(mixture).unsqueeze(0)],
                      descriptions=[prompt]).to("cuda")

    # Two draws back to back: the first includes any lazy allocation, the second is steady state.
    timings = []
    for _ in range(2):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.monotonic()
        with torch.inference_mode():
            model.separate(batch, predict_spans=False, reranking_candidates=1)
        torch.cuda.synchronize()
        timings.append((round(time.monotonic() - started, 1),
                        round(gib(torch.cuda.max_memory_allocated()), 2)))
    results["draw_seconds"] = [t for t, _ in timings]
    results["peak_during_draw_gib"] = max(peak for _, peak in timings)
    results["fits_24_gib_gpu"] = results["peak_during_draw_gib"] < L4_LIMIT_GIB
    steady = timings[-1][0]
    results["gpu_seconds_per_isolation"] = steady
    results["dollars_per_isolation"] = {
        name: round(steady * rate, 4) for name, rate in MODAL_RATES_PER_SECOND.items()
    }
    print("PERF " + json.dumps(results))


if __name__ == "__main__":
    main()
'''

Path("/content/perf_no_judge.py").write_text(PERF_NO_JUDGE)
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python",
     "/content/perf_no_judge.py", str(perf_case), MODEL_ID, "cello"],
    env=ENV, capture_output=True, text=True, timeout=1800,
)
line = [l for l in done.stdout.splitlines() if l.startswith("PERF ")]
if not line:
    print("exit", done.returncode)
    print(done.stdout[-1200:])
    print("--- stderr ---")
    print(done.stderr[-2500:])
else:
    no_judge = json.loads(line[0].removeprefix("PERF "))
    (BASE / "runs" / "holdout-004" / "performance-no-judge.json").write_text(
        json.dumps(no_judge, indent=2)
    )
    for key, value in no_judge.items():
        print(f"  {key:36s} {value}")


  model_load_seconds                   40.5
  model_resident_gib                   14.32
  draw_seconds                         [11.0, 10.3]
  peak_during_draw_gib                 22.42
  fits_24_gib_gpu                      True
  gpu_seconds_per_isolation            10.3
  dollars_per_isolation                {'A100-40GB': 0.006, 'L4': 0.0023}


In [ ]:
"""M5c: stage wall-clock for the held-out run, to cost the CPU Demucs path per isolation."""

cases = len(json.loads((BASE / "runs" / "holdout-004" / "config.json").read_text())["cases"])
print(f"holdout-004: {cases} cases\n")
print(f"  {'stage':12s} {'status':10s} {'total s':>9s} {'per case s':>11s}")
print("  " + "-" * 45)
for line in (BASE / "runs" / "holdout-004" / "stage-times.jsonl").read_text().splitlines():
    entry = json.loads(line)
    print(f"  {entry['stage']:12s} {entry['status']:10s} {entry['seconds']:9.1f} "
          f"{entry['seconds'] / cases:11.1f}")


holdout-004: 10 cases

  stage        status       total s  per case s
  ---------------------------------------------
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/mixray-quality/runs/holdout-004/stage-times.jsonl'


In [ ]:
"""M5c: time the CPU Demucs path directly, since it serves the common case after routing."""

import shutil
import time

probe_dir = Path("/content/demucs-timing")
shutil.rmtree(probe_dir, ignore_errors=True)
probe_dir.mkdir(parents=True)
clip = BASE / "runs" / "holdout-004" / "nice_guy_vox" / "mix.wav"
shutil.copy2(clip, probe_dir / "mix.wav")

argv = [UV, "run", "--no-sync", "--project", str(PROJECTS["demucs"]), "python", "-m",
        "demucs.separate", "--name", "htdemucs_6s", "--device", "cpu",
        "--out", str(probe_dir / "out"), "--filename", "{stem}.{ext}",
        str(probe_dir / "mix.wav")]
demucs_env = dict(ENV)
demucs_env["CUDA_VISIBLE_DEVICES"] = "-1"

# First call includes the model download, so a second call is the figure production would see.
timings = []
for attempt in range(2):
    started = time.monotonic()
    done = subprocess.run(argv, env=demucs_env, capture_output=True, text=True, timeout=1800)
    timings.append(round(time.monotonic() - started, 1))
    if done.returncode:
        print("exit", done.returncode)
        print(done.stderr[-2000:])
        break

produced = sorted(p.name for p in (probe_dir / "out" / "htdemucs_6s" / "mix").glob("*.wav"))
print(f"  cold then warm seconds: {timings}")
print(f"  stems produced:         {produced}")
print(f"  seconds per clip (warm, all six stems at once): {timings[-1]}")
print("  every standard prompt for this clip is then served from cache with no model at all")


  cold then warm seconds: [14.3, 14.0]
  stems produced:         []
  seconds per clip (warm, all six stems at once): 14.0
  every standard prompt for this clip is then served from cache with no model at all


In [ ]:
for path in sorted(Path("/content/demucs-timing/out").rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to('/content/demucs-timing/out')}  "
              f"{path.stat().st_size / 2**20:.1f} MiB")


  htdemucs_6s/bass.wav  5.0 MiB
  htdemucs_6s/drums.wav  5.0 MiB
  htdemucs_6s/guitar.wav  5.0 MiB
  htdemucs_6s/other.wav  5.0 MiB
  htdemucs_6s/piano.wav  5.0 MiB
  htdemucs_6s/vocals.wav  5.0 MiB


In [ ]:
"""Correcting this morning's VRAM figure: allocated understates what a card must actually have."""

RESERVED = r'''
"""Report allocated, reserved and the driver's own view, which is what decides whether it fits."""

import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf

RATE = 48000
RANKERS_OFF = {"visual_ranker": None, "text_ranker": None, "span_predictor": None}
L4_TOTAL_GIB = 22.49  # what nvidia-smi reports for a 24 GB L4


def gib(value):
    return round(value / 2**30, 2)


def driver_used_gib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True).stdout.strip()
    return round(int(out.splitlines()[0]) / 1024, 2)


def main():
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor

    case_dir, model_id, prompt = Path(sys.argv[1]), sys.argv[2], sys.argv[3]
    revision = json.loads(
        (case_dir / model_id.split("/")[-1] / "model.json").read_text()
    )["revision"]
    snapshot = snapshot_download(model_id, revision=revision,
                                 allow_patterns=["*.json", "*.pt", "*.safetensors"])
    model = SAMAudio.from_pretrained(snapshot, **RANKERS_OFF).eval().cuda()
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    audio, rate = sf.read(str(case_dir / "mix.wav"), dtype="float64", always_2d=True)
    mixture = np.asarray(audio.mean(axis=1), dtype=np.float32)
    batch = processor(audios=[torch.tensor(mixture).unsqueeze(0)],
                      descriptions=[prompt]).to("cuda")

    results = {}
    for attempt in range(2):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.monotonic()
        with torch.inference_mode():
            model.separate(batch, predict_spans=False, reranking_candidates=1)
        torch.cuda.synchronize()
        results = {
            "draw_seconds": round(time.monotonic() - started, 1),
            "peak_allocated_gib": gib(torch.cuda.max_memory_allocated()),
            "peak_reserved_gib": gib(torch.cuda.max_memory_reserved()),
            "driver_reports_used_gib": driver_used_gib(),
        }
    results["l4_total_gib"] = L4_TOTAL_GIB
    results["headroom_on_l4_gib"] = round(L4_TOTAL_GIB - results["driver_reports_used_gib"], 2)
    results["fits_l4"] = results["headroom_on_l4_gib"] > 0
    print("RESERVED " + json.dumps(results))


if __name__ == "__main__":
    main()
'''

Path("/content/reserved.py").write_text(RESERVED)
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["sam"]), "python",
     "/content/reserved.py", str(perf_case), MODEL_ID, "cello"],
    env=ENV, capture_output=True, text=True, timeout=1800,
)
line = [l for l in done.stdout.splitlines() if l.startswith("RESERVED ")]
if not line:
    print("exit", done.returncode)
    print(done.stdout[-800:])
    print(done.stderr[-2000:])
else:
    for key, value in json.loads(line[0].removeprefix("RESERVED ")).items():
        print(f"  {key:28s} {value}")


  draw_seconds                 10.3
  peak_allocated_gib           22.42
  peak_reserved_gib            24.22
  driver_reports_used_gib      24.73
  l4_total_gib                 22.49
  headroom_on_l4_gib           -2.24
  fits_l4                      False


## Demucs `other` for organ, synth and strings


In [ ]:
"""Would routing synth, organ, strings to Demucs' `other` stem beat SAM Audio's 0.007?

`other` currently has no keywords, so no prompt can reach it. These four cases are exactly the
population it would serve.
"""

OTHER = r'''
"""N1 for the `other` stem on the cases whose prompt reaches no Demucs category today."""

import json
import sys
import tempfile
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000
UNCOVERED = {"diva_synth", "evening_organ", "mend_cello", "summer_synth"}


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def f1(reference_events, events):
    if not reference_events or not events:
        return 0.0
    reference = np.array([[e[0], e[1]] for e in reference_events], dtype=float).reshape(-1, 2)
    estimate = np.array([[e[0], e[1]] for e in events], dtype=float).reshape(-1, 2)
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        reference, hz([e[2] for e in reference_events]),
        estimate, hz([e[2] for e in events]),
        onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None,
    )
    return float(value)


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    model = Model(ICASSP_2022_MODEL_PATH)
    rows = []
    with tempfile.TemporaryDirectory() as scratch:
        staging = Path(scratch) / "clip.wav"

        def notes(samples):
            peak = float(np.max(np.abs(samples)))
            scaled = samples * (0.891 / peak) if peak > 0.01 else samples
            sf.write(str(staging), scaled.astype(np.float32), RATE, subtype="FLOAT")
            _, _, detected = predict(staging, model, onset_threshold=0.5, frame_threshold=0.3,
                                     minimum_note_length=30.0, melodia_trick=True)
            return [(e[0], e[1], e[2]) for e in detected]

        for run_dir in sys.argv[1:]:
            run = Path(run_dir)
            for case in json.loads((run / "config.json").read_text())["cases"]:
                if case["id"] not in UNCOVERED:
                    continue
                case_dir = run / case["id"]
                reference = mono(case_dir / "reference.wav")
                length = len(reference)
                reference_events = notes(reference)
                row = {"case": case["id"], "prompt": case["prompt"],
                       "reference_notes": len(reference_events)}
                mixture = mono(case_dir / "mix.wav", length)
                mix_rms = float(np.sqrt(np.mean(mixture**2)))
                row["mix"] = f1(reference_events, notes(mixture))
                for name in ("other", "piano", "guitar"):
                    path = case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{name}.wav"
                    if not path.is_file():
                        continue
                    served = mono(path, length)
                    rms = float(np.sqrt(np.mean(served**2)))
                    row[name] = f1(reference_events, notes(served))
                    row[name + "_level_db"] = 20.0 * np.log10(
                        max(rms, 1e-12) / max(mix_rms, 1e-12)
                    )
                candidate = case_dir / "sam-audio-large" / "candidate-0.wav"
                if candidate.is_file():
                    row["sam"] = f1(reference_events, notes(mono(candidate, length)))
                rows.append(row)
                print(f"  {case['id']}", flush=True)
    Path("/content/other_routing.json").write_text(json.dumps(rows, indent=2))
    print("OTHER_DONE " + json.dumps({"cases": len(rows)}))


if __name__ == "__main__":
    main()
'''

Path("/content/other_routing.py").write_text(OTHER)
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/other_routing.py", *[str(BASE / "runs" / n)
                                    for n in ("dev-002", "holdout-004")]],
    env=gate_env, capture_output=True, text=True, timeout=1800,
)
if done.returncode:
    print(done.stdout[-800:])
    print(done.stderr[-2500:])
else:
    rows = json.loads(Path("/content/other_routing.json").read_text())
    print(f"\n  {'case':16s} {'prompt':14s} {'mix':>6s} {'SAM':>6s} {'other':>6s} "
          f"{'other dB':>9s} {'piano':>6s} {'guitar':>7s}")
    print("  " + "-" * 76)
    for row in rows:
        print(f"  {row['case']:16s} {row['prompt']:14s} {row['mix']:6.3f} "
              f"{row.get('sam', float('nan')):6.3f} {row.get('other', float('nan')):6.3f} "
              f"{row.get('other_level_db', float('nan')):9.1f} "
              f"{row.get('piano', float('nan')):6.3f} {row.get('guitar', float('nan')):7.3f}")
    from statistics import mean
    print("  " + "-" * 76)
    for key in ("mix", "sam", "other", "piano", "guitar"):
        values = [r[key] for r in rows if key in r]
        if values:
            print(f"  mean {key:10s} {mean(values):.3f}")


  case             prompt            mix    SAM  other  other dB  piano  guitar
  ----------------------------------------------------------------------------
  diva_synth       synthesizer     0.000  0.000  0.033     -39.9  0.013   0.034
  evening_organ    organ           0.080  0.035  0.268     -22.3  0.154   0.099
  summer_synth     synthesizer     0.000  0.000  0.135     -35.0  0.005   0.013
  mend_cello       cello           0.065  0.000  0.101      -7.5  0.062   0.089
  ----------------------------------------------------------------------------
  mean mix        0.036
  mean sam        0.009
  mean other      0.134
  mean piano      0.059
  mean guitar     0.059


In [ ]:
"""Record the day's decisions and the resulting product shape beside the runs."""

RESULTS = """# Results — 2026-11-01

All measured on this Colab VM. Note figures are agreement with the decoder's own output on the
clean stem, except N3, which is against GuitarSet's published labels.

## Routing, measured over 20 targets in 10 AlbumDB songs

| population | cases | raw mixture | SAM Audio | Demucs |
|---|---:|---:|---:|---:|
| prompt names one of the five reachable categories | 16 | 0.271 | 0.388 | 0.532 |
| prompt reaches no category (organ, cello, 2 synths) | 4 | 0.036 | 0.009 | 0.134 via `other` |

Demucs leads SAM by 0.144 where both apply. Opening the `other` category lifts the second
population from SAM's 0.009 to 0.134, clearing the same 0.05 bar against the raw mixture.

## Decoder, measured over 360 GuitarSet excerpts with real labels

Split by performer: players 00-02 develop, 03-05 hold out. 55,777 labelled notes.

| setting | develop F1 | holdout F1 | notes made / notes real |
|---|---:|---:|---:|
| 30 ms minimum note | 0.719 | 0.732 | 1.40 |
| 127.70 ms minimum note | 0.766 | **0.777** | **1.02** |

The decoder's weakness is invention, not deafness: recall 0.880, precision 0.628. The longer gate
trades 9 points of recall for 14 of precision.

Applied to guitar only. On the shipped set it gained guitar 0.036 and cost vocals 0.043 and bass
0.017, so a blanket change failed the acceptance rule.

| class | n | before | after |
|---|---:|---:|---:|
| bass | 2 | 0.745 | 0.745 |
| vocals | 2 | 0.685 | 0.685 |
| guitar | 8 | 0.634 | **0.671** |
| piano | 3 | 0.184 | 0.184 |
| all five categories | 15 | 0.566 | **0.585** |

## The absent-sound gate

120 stems from 20 mixtures. Every one of the 31 instruments absent from its mixture landed between
-46.4 and -37.0 dB under it; 81 of 89 present stems stayed above -25 dB.

| gate at -25 dB | result |
|---|---|
| absent stems caught | 31 / 31 |
| deliverable stems passed | 81 / 89 |
| present but below the bar | 8, none of them usable |

The eight include the piano Demucs never found, whose notes scored 0.015 and 0.076, and vocals in
windows where the singer is not performing. None is a false alarm in product terms.

The gate catches "there is nothing here". It does not catch "this instrument defeats the decoder":
the cello stem sits at -7.5 dB, clearly present, and still transcribes at 0.101.

## Shipped behaviour after today

| request | served by | measured notes |
|---|---|---|
| vocals, bass, guitar, piano | Demucs stem, no GPU, 14.0 s for all six at once, cached | 0.585 over 15 cases |
| drums | Demucs stem, pitch transcription skipped by design | not applicable |
| synth, organ, strings, brass, woodwind, keys | Demucs `other` | 0.134 over 4 cases |
| anything else | SAM Audio, one draw, A100-40GB, 10.3 s, $0.0060 | 0.009 over 4 cases |
| requested sound more than 25 dB under the mixture | refused, with the reason | n/a |

## Corrections made today

- `large` does **not** fit a 24 GB L4. This morning's 22.42 GiB was `max_memory_allocated`;
  reserved is 24.22 GiB and the driver reports 24.73 GiB in use against an L4's 22.49 GiB.
- The cross-check selector proposed yesterday was rejected: +0.005 mean, five cases worse.
- Judge's `precision` axis is not a wrong-source signal. Against one request it scored the bass
  4.955, the true piano 4.553 and Demucs' piano stem 1.490.

## Open, with reasons

- Piano at 0.184 and the `other` classes at 0.134 are separation limits, not decoder settings.
  Nothing in the permitted toolset addresses them.
- The decoder ceiling on clean guitar is 0.777, so roughly a fifth is lost before separation.
- bass and vocals rest on two cases each; only guitar has two independent datasets behind it.
- Real bass recordings prefer 30 ms over the current 127.70 ms by 0.016, which contradicts the
  synthetic curve the constant came from. Too small to act on, worth revisiting with more data.
"""
(BASE / "runs" / "RESULTS-2026-11-01.md").write_text(RESULTS)
print(f"written to {BASE / 'runs' / 'RESULTS-2026-11-01.md'}")
print(f"{len(RESULTS.splitlines())} lines")


written to /content/drive/MyDrive/mixray-quality/runs/RESULTS-2026-11-01.md
81 lines


## Demucs first, then SAM Audio


In [ ]:
"""C1: Demucs strips bass and drums first, then SAM Audio isolates from what is left.

SAM's measured failure is returning bass for most prompts. With bass removed before it runs, that
failure is unavailable to it. Tested on the two worst classes, against Demucs alone as the bar.
Variant A keeps vocals in SAM's input; variant B removes them too.
"""

CASCADE_SEPARATE = r'''
import json
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
RANKERS_OFF = {"visual_ranker": None, "text_ranker": None, "span_predictor": None}
KEEP = {"A": ("vocals", "guitar", "piano", "other"), "B": ("guitar", "piano", "other")}


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def main():
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor

    spec = json.loads(Path(sys.argv[1]).read_text())
    out_root = Path(sys.argv[2])
    first = Path(spec[0]["dir"])
    revision = json.loads((first / "sam-audio-large" / "model.json").read_text())["revision"]
    snapshot = snapshot_download("facebook/sam-audio-large", revision=revision,
                                 allow_patterns=["*.json", "*.pt", "*.safetensors"])
    model = SAMAudio.from_pretrained(snapshot, **RANKERS_OFF).eval().cuda()
    processor = SAMAudioProcessor.from_pretrained(snapshot)
    for case in spec:
        case_dir = Path(case["dir"])
        length = len(mono(case_dir / "reference.wav"))
        stems = {s: mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{s}.wav", length)
                 for s in ("vocals", "drums", "bass", "guitar", "piano", "other")}
        target = out_root / case["id"]
        target.mkdir(parents=True, exist_ok=True)
        for variant, kept in KEEP.items():
            source = sum(stems[s] for s in kept)
            peak = float(np.max(np.abs(source)))
            source = source * (0.891 / peak) if peak > 0 else source
            batch = processor(audios=[torch.tensor(source, dtype=torch.float32).unsqueeze(0)],
                              descriptions=[case["prompt"]]).to("cuda")
            torch.manual_seed(1000)
            torch.cuda.manual_seed_all(1000)
            with torch.inference_mode():
                result = model.separate(batch, predict_spans=False, reranking_candidates=1)
            drawn = result.target[0].detach().float().cpu().numpy().reshape(-1)[:length]
            sf.write(str(target / f"cascade-{variant}.wav"), drawn, RATE, subtype="FLOAT")
        print(f"  separated {case['id']}", flush=True)
    print("CASCADE_SEPARATED")


if __name__ == "__main__":
    main()
'''

CASCADE_SCORE = r'''
import json
import sys
import tempfile
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def si_sdr(reference, estimate):
    n = min(len(reference), len(estimate))
    t, a = reference[:n] - reference[:n].mean(), estimate[:n] - estimate[:n].mean()
    p = float(t @ t)
    if p < 1e-12:
        return float("nan")
    proj = t * float(a @ t) / p
    err = a - proj
    return 10 * np.log10((proj @ proj + 1e-12) / (err @ err + 1e-12))


def hz(values):
    return 440.0 * 2 ** ((np.asarray(values, dtype=float) - 69) / 12)


def f1(reference_events, events):
    if not reference_events or not events:
        return 0.0
    r = np.array([[e[0], e[1]] for e in reference_events]).reshape(-1, 2)
    e = np.array([[x[0], x[1]] for x in events]).reshape(-1, 2)
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        r, hz([x[2] for x in reference_events]), e, hz([x[2] for x in events]),
        onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None)
    return float(value)


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    spec = json.loads(Path(sys.argv[1]).read_text())
    out_root = Path(sys.argv[2])
    model = Model(ICASSP_2022_MODEL_PATH)
    rows = []
    with tempfile.TemporaryDirectory() as scratch:
        staging = Path(scratch) / "clip.wav"

        def notes(samples):
            peak = float(np.max(np.abs(samples)))
            scaled = samples * (0.891 / peak) if peak > 0.01 else samples
            sf.write(str(staging), scaled.astype(np.float32), RATE, subtype="FLOAT")
            _, _, found = predict(staging, model, onset_threshold=0.5, frame_threshold=0.3,
                                  minimum_note_length=30.0, melodia_trick=True)
            return [(x[0], x[1], x[2]) for x in found]

        for case in spec:
            case_dir = Path(case["dir"])
            reference = mono(case_dir / "reference.wav")
            n = len(reference)
            truth = notes(reference)
            mixture = mono(case_dir / "mix.wav", n)
            served = mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{case['stem']}.wav", n)
            row = {"case": case["id"], "prompt": case["prompt"], "stem": case["stem"]}
            candidates = {"mixture": mixture, f"demucs {case['stem']}": served}
            for variant in ("A", "B"):
                path = out_root / case["id"] / f"cascade-{variant}.wav"
                if path.is_file():
                    candidates[f"cascade {variant}"] = mono(path, n)
            for name, samples in candidates.items():
                row[name] = {"si_sdr": si_sdr(reference, samples) - si_sdr(reference, mixture),
                             "n1": f1(truth, notes(samples))}
            rows.append(row)
    Path("/content/cascade.json").write_text(json.dumps(rows, indent=2))
    print("CASCADE_SCORED")


if __name__ == "__main__":
    main()
'''

CASES_C1 = [
    ("mag_piano_sparse", "dev-002", "piano", "piano"),
    ("mag_piano_full", "dev-002", "piano", "piano"),
    ("karma_piano", "holdout-004", "piano", "piano"),
    ("evening_organ", "holdout-004", "organ", "other"),
    ("mend_cello", "holdout-004", "cello", "other"),
    ("summer_synth", "holdout-004", "synthesizer", "other"),
    ("diva_synth", "dev-002", "synthesizer", "other"),
]
spec = [{"id": i, "dir": str(BASE / "runs" / r / i), "prompt": p, "stem": s}
        for i, r, p, s in CASES_C1]
Path("/content/cascade_spec.json").write_text(json.dumps(spec))
Path("/content/cascade_separate.py").write_text(CASCADE_SEPARATE)
Path("/content/cascade_score.py").write_text(CASCADE_SCORE)
CASCADE_OUT = Path("/content/cascade")

lines = ["set -o pipefail", "echo C1_START",
         f"{UV} run --no-sync --project {PROJECTS['sam']} python /content/cascade_separate.py "
         f"/content/cascade_spec.json {CASCADE_OUT} || exit 1",
         f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['pitch']} python "
         f"/content/cascade_score.py /content/cascade_spec.json {CASCADE_OUT} || exit 1",
         "echo C1_DONE"]
script = WORK / "cascade.sh"
script.write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/cascade.log", "/content/cascade.pid", "C1_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"cascade over {len(spec)} cases, two variants each, as pid {job.pid}")


cascade over 7 cases, two variants each, as pid 35760


In [ ]:
"""C1 results: cascade against Demucs alone, separation (S1) and notes (N1)."""

from statistics import mean

cascade = json.loads(Path("/content/cascade.json").read_text())
columns = ["mixture", "demucs", "cascade A", "cascade B"]
print("N1 (notes) and S1 (dB over the mixture); A keeps vocals in SAM's input, B removes them\n")
print(f"  {'case':18s} {'prompt':12s}" + "".join(f"{c:>17s}" for c in columns))
print("  " + "-" * 98)
totals = {c: [] for c in columns}
for row in cascade:
    cells = []
    for column in columns:
        key = next((k for k in row if k.startswith(column)), None)
        if key is None:
            cells.append(f"{'-':>17s}")
            continue
        value = row[key]
        totals[column].append(value["n1"])
        cells.append(f"   {value['n1']:.3f} {value['si_sdr']:+6.1f} dB")
    print(f"  {row['case']:18s} {row['prompt']:12s}" + "".join(cells))
print("  " + "-" * 98)
print(f"  {'mean N1':31s}" + "".join(f"{mean(totals[c]):>17.3f}" for c in columns))

for population, names in (("piano", {"mag_piano_sparse", "mag_piano_full", "karma_piano"}),
                          ("other", {"evening_organ", "mend_cello", "summer_synth", "diva_synth"})):
    subset = [r for r in cascade if r["case"] in names]
    line = []
    for column in columns:
        values = [r[next(k for k in r if k.startswith(column))]["n1"] for r in subset
                  if any(k.startswith(column) for k in r)]
        line.append(f"{column} {mean(values):.3f}")
    print(f"\n  {population:6s} " + "   ".join(line))


N1 (notes) and S1 (dB over the mixture); A keeps vocals in SAM's input, B removes them

  case               prompt                mixture           demucs        cascade A        cascade B
  --------------------------------------------------------------------------------------------------
  mag_piano_sparse   piano          0.008   +0.0 dB   0.076   +6.4 dB   0.067   +5.9 dB   0.006   -5.2 dB
  mag_piano_full     piano          0.021   +0.0 dB   0.015   -6.5 dB   0.036   -0.1 dB   0.023   -2.5 dB
  karma_piano        piano          0.177   +0.0 dB   0.460   +9.5 dB   0.237   -0.9 dB   0.250   -0.9 dB
  evening_organ      organ          0.080   +0.0 dB   0.268  +23.4 dB   0.000  -78.5 dB   0.000  -77.5 dB
  mend_cello         cello          0.065   +0.0 dB   0.101   +5.9 dB   0.000  -66.7 dB   0.024  -26.4 dB
  summer_synth       synthesizer    0.000   +0.0 dB   0.135  +18.5 dB   0.000  -62.0 dB   0.000  -62.1 dB
  diva_synth         synthesizer    0.000   +0.0 dB   0.033  +18.9 dB   0

## `htdemucs_ft` and shifts


In [ ]:
"""S2: can Demucs itself do better? Two untried options, scored on notes against the current output.

htdemucs_ft is Demucs' fine-tuned four-source bag, documented as its strongest on vocals, bass and
drums; it has no piano or guitar, so it is tried only where it applies. --shifts averages several
randomly time-shifted passes.
"""

S2_SEPARATE = r'''
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

spec = json.loads(Path(sys.argv[1]).read_text())
out_root = Path(sys.argv[2])
VARIANTS = {"vocals": [("ft", ["-n", "htdemucs_ft"])],
            "bass": [("ft", ["-n", "htdemucs_ft"])],
            "piano": [("6s_shifts4", ["-n", "htdemucs_6s", "--shifts", "4"])],
            "guitar": [("6s_shifts4", ["-n", "htdemucs_6s", "--shifts", "4"])]}
timings = {}
for case in spec:
    for name, flags in VARIANTS[case["stem"]]:
        target = out_root / case["id"] / name
        if (target / f"{case['stem']}.wav").is_file():
            continue
        target.mkdir(parents=True, exist_ok=True)
        started = time.monotonic()
        subprocess.run([sys.executable, "-m", "demucs.separate", *flags, "--device", "cpu",
                        "--out", str(target / "raw"), "--filename", "{stem}.{ext}",
                        str(Path(case["dir"]) / "mix.wav")], check=True,
                       capture_output=True)
        timings.setdefault(name, []).append(round(time.monotonic() - started, 1))
        produced = next((target / "raw").rglob(f"{case['stem']}.wav"))
        shutil.copy2(produced, target / f"{case['stem']}.wav")
        print(f"  {case['id']} {name} {timings[name][-1]} s", flush=True)
Path("/content/s2_timings.json").write_text(json.dumps(timings))
print("S2_SEPARATED")
'''

S2_SCORE = r'''
import json
import sys
import tempfile
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000
GUITAR_MS, DEFAULT_MS = 127.7, 30.0


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def hz(v):
    return 440.0 * 2 ** ((np.asarray(v, dtype=float) - 69) / 12)


def f1(truth, events):
    if not truth or not events:
        return 0.0
    r = np.array([[e[0], e[1]] for e in truth]).reshape(-1, 2)
    e = np.array([[x[0], x[1]] for x in events]).reshape(-1, 2)
    _, _, value, _ = mir_eval.transcription.precision_recall_f1_overlap(
        r, hz([x[2] for x in truth]), e, hz([x[2] for x in events]),
        onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None)
    return float(value)


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    spec = json.loads(Path(sys.argv[1]).read_text())
    out_root = Path(sys.argv[2])
    model = Model(ICASSP_2022_MODEL_PATH)
    rows = []
    with tempfile.TemporaryDirectory() as scratch:
        staging = Path(scratch) / "clip.wav"

        def notes(samples, ms):
            peak = float(np.max(np.abs(samples)))
            scaled = samples * (0.891 / peak) if peak > 0.01 else samples
            sf.write(str(staging), scaled.astype(np.float32), RATE, subtype="FLOAT")
            _, _, found = predict(staging, model, onset_threshold=0.5, frame_threshold=0.3,
                                  minimum_note_length=ms, melodia_trick=True)
            return [(x[0], x[1], x[2]) for x in found]

        for case in spec:
            ms = GUITAR_MS if case["stem"] == "guitar" else DEFAULT_MS
            case_dir = Path(case["dir"])
            reference = mono(case_dir / "reference.wav")
            n = len(reference)
            truth = notes(reference, ms)
            current = mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{case['stem']}.wav", n)
            row = {"case": case["id"], "stem": case["stem"],
                   "current 6s": f1(truth, notes(current, ms))}
            for variant_dir in sorted((out_root / case["id"]).iterdir()):
                path = variant_dir / f"{case['stem']}.wav"
                if path.is_file():
                    row[variant_dir.name] = f1(truth, notes(mono(path, n), ms))
            rows.append(row)
    Path("/content/s2.json").write_text(json.dumps(rows, indent=2))
    print("S2_SCORED")


if __name__ == "__main__":
    main()
'''

CASES_S2 = [
    ("ex_bass", "dev-002", "bass"), ("evening_bass", "holdout-004", "bass"),
    ("mend_vox", "holdout-004", "vocals"), ("nice_guy_vox", "holdout-004", "vocals"),
    ("mag_piano_sparse", "dev-002", "piano"), ("karma_piano", "holdout-004", "piano"),
    ("light_guitar_solo", "dev-003", "guitar"), ("seas_guitar_electric", "dev-003", "guitar"),
    ("light_guitar_rhythm", "dev-003", "guitar"),
]
s2_spec = [{"id": i, "dir": str(BASE / "runs" / r / i), "stem": s} for i, r, s in CASES_S2]
Path("/content/s2_spec.json").write_text(json.dumps(s2_spec))
Path("/content/s2_separate.py").write_text(S2_SEPARATE)
Path("/content/s2_score.py").write_text(S2_SCORE)
S2_OUT = Path("/content/s2")

lines = ["set -o pipefail", "echo S2_START",
         f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['demucs']} python "
         f"/content/s2_separate.py /content/s2_spec.json {S2_OUT} || exit 1",
         f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['pitch']} python "
         f"/content/s2_score.py /content/s2_spec.json {S2_OUT} || exit 1",
         "echo S2_DONE"]
script = WORK / "s2.sh"
script.write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/s2.log", "/content/s2.pid", "S2_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"S2 over {len(s2_spec)} cases as pid {job.pid}")


S2 over 9 cases as pid 37168


In [ ]:
"""S2 results: per case and per class, against the current htdemucs_6s output."""

from statistics import mean

s2 = json.loads(Path("/content/s2.json").read_text())
timing = json.loads(Path("/content/s2_timings.json").read_text())
print(f"  {'case':22s} {'stem':7s} {'current 6s':>11s} {'variant':>12s} {'N1':>7s} {'delta':>7s}")
print("  " + "-" * 72)
by_class = {}
for row in s2:
    variant = next(k for k in row if k not in {"case", "stem", "current 6s"})
    delta = row[variant] - row["current 6s"]
    by_class.setdefault((row["stem"], variant), []).append((row["current 6s"], row[variant]))
    print(f"  {row['case']:22s} {row['stem']:7s} {row['current 6s']:11.3f} {variant:>12s} "
          f"{row[variant]:7.3f} {delta:+7.3f}")
print()
for (stem, variant), pairs in by_class.items():
    before, after = mean(p[0] for p in pairs), mean(p[1] for p in pairs)
    seconds = mean(timing.get(variant, [0.0]))
    print(f"  {stem:7s} {variant:12s} n={len(pairs)}  {before:.3f} -> {after:.3f}  "
          f"({after - before:+.3f})  {seconds:.0f} s per clip vs 14 s now")


  case                   stem     current 6s      variant      N1   delta
  ------------------------------------------------------------------------
  ex_bass                bass          0.759           ft   0.751  -0.008
  evening_bass           bass          0.732           ft   0.751  +0.019
  mend_vox               vocals        0.711           ft   0.714  +0.002
  nice_guy_vox           vocals        0.659           ft   0.731  +0.072
  mag_piano_sparse       piano         0.076   6s_shifts4   0.076  +0.001
  karma_piano            piano         0.460   6s_shifts4   0.451  -0.009
  light_guitar_solo      guitar        0.449   6s_shifts4   0.461  +0.012
  seas_guitar_electric   guitar        0.389   6s_shifts4   0.408  +0.019
  light_guitar_rhythm    guitar        0.605   6s_shifts4   0.573  -0.033

  bass    ft           n=2  0.745 -> 0.751  (+0.006)  59 s per clip vs 14 s now
  vocals  ft           n=2  0.685 -> 0.722  (+0.037)  59 s per clip vs 14 s now
  piano   6s_shifts4   n

In [ ]:
"""V1: vocals and bass rest on two cases each. Every AlbumDB song has both, so take all ten.

One mixture per song, two targets sharing its offset. Built with the same worker prepare stage as
every earlier run, so mixing and gain match. Then htdemucs_6s (shipped) against htdemucs_ft.
"""

RUN_ID = "voxbass-006"
RUN = BASE / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)

GROUPS = {}
for song_dir in sorted(SONGS.iterdir()):
    names = sorted(p.name for p in song_dir.glob("*.wav"))
    chosen = ["vox_lead.wav", "bass.wav"]
    for prefix in ("drums_kick", "drums_overheads", "guitar_electric", "guitar_acoustic",
                   "keys_"):
        match = next((n for n in names if n.startswith(prefix) and n not in chosen), None)
        if match and len(chosen) < 6:
            chosen.append(match)
    key = song_dir.name.split(" ")[0]
    GROUPS[key] = {"song": song_dir.name,
                   "stems": [str(song_dir / n) for n in chosen],
                   "cases": [(f"s{key}_vox", "vocals", "lead vocals", "vox_lead.wav"),
                             (f"s{key}_bass", "bass", "bass guitar", "bass.wav")]}

request = {k: {"stems": g["stems"],
               "targets": [str(SONGS / g["song"] / t) for *_, t in g["cases"]]}
           for k, g in GROUPS.items()}
completed = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "-c", ENERGY_SCAN],
    input=json.dumps(request), env=scan_env, capture_output=True, text=True, timeout=1800,
)
if completed.returncode:
    raise RuntimeError(completed.stderr[-2000:])
offsets = json.loads(next(l for l in completed.stdout.splitlines()
                          if l.startswith("ENERGY_SCAN ")).removeprefix("ENERGY_SCAN "))
(RUN / "offsets.json").write_text(json.dumps(offsets, indent=2))

CASES = []
for key, group in GROUPS.items():
    picked = offsets[key]
    for identifier, instrument, prompt, target in group["cases"]:
        path = str(SONGS / group["song"] / target)
        print(f"  {identifier:10s} start {picked['start']:5.0f} s  "
              f"{picked['levels'][path]['to_mix_db']:+6.1f} dB  stems {len(group['stems'])}")
        CASES.append({"id": identifier, "song": group["song"], "split": "dev",
                      "instrument": instrument, "prompt": prompt,
                      "source_url": "https://zenodo.org/records/19683001",
                      "license": "CC-BY-4.0", "stems": group["stems"], "targets": [path],
                      "start": picked["start"], "duration": 30.0, "notes": None})
config_path = RUN / "config.json"
config_path.write_text(json.dumps({"output": str(RUN), "model": MODEL_ID, "draws": 1,
                                   "allow_holdout": False, "cases": CASES}, indent=2))

lines = ["set -o pipefail", "echo V1_START"]
for stage, project in (("prepare", "pitch"), ("demucs", "demucs")):
    lines.append(f'echo "--- {stage} ---"')
    lines.append(f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS[project]} "
                 f"python {WORKER} {stage} {config_path} || exit 1")
v1_spec = [{"id": c["id"], "dir": str(RUN / c["id"]), "stem": c["instrument"]} for c in CASES]
Path("/content/v1_spec.json").write_text(json.dumps(v1_spec))
VARIANT_FT = Path("/content/s2_separate.py").read_text().replace(
    '"piano": [("6s_shifts4", ["-n", "htdemucs_6s", "--shifts", "4"])],\n'
    '            "guitar": [("6s_shifts4", ["-n", "htdemucs_6s", "--shifts", "4"])]}',
    '}').replace('Path("/content/s2_timings.json")', 'Path("/content/v1_timings.json")')
Path("/content/v1_separate.py").write_text(VARIANT_FT)
Path("/content/v1_score.py").write_text(
    Path("/content/s2_score.py").read_text().replace('"/content/s2.json"', '"/content/v1.json"'))
lines += ['echo "--- htdemucs_ft ---"',
          f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['demucs']} python "
          f"/content/v1_separate.py /content/v1_spec.json /content/v1 || exit 1",
          'echo "--- score ---"',
          f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['pitch']} python "
          f"/content/v1_score.py /content/v1_spec.json /content/v1 || exit 1",
          "echo V1_DONE"]
script = WORK / "v1.sh"
script.write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/v1.log", "/content/v1.pid", "V1_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(script)], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"\n{len(CASES)} cases (10 vocals, 10 bass) as pid {job.pid}")


  s01_vox    start    46 s    -5.2 dB  stems 6
  s01_bass   start    46 s    -5.3 dB  stems 6
  s02_vox    start   165 s    -4.6 dB  stems 6
  s02_bass   start   165 s    -4.2 dB  stems 6
  s03_vox    start   125 s    -3.6 dB  stems 6
  s03_bass   start   125 s    -4.3 dB  stems 6
  s04_vox    start   211 s    -5.3 dB  stems 6
  s04_bass   start   211 s    -3.1 dB  stems 6
  s05_vox    start   141 s    -5.4 dB  stems 6
  s05_bass   start   141 s    -3.3 dB  stems 6
  s06_vox    start   141 s    -4.4 dB  stems 6
  s06_bass   start   141 s    -7.3 dB  stems 6
  s07_vox    start   130 s    -4.7 dB  stems 6
  s07_bass   start   130 s    -6.4 dB  stems 6
  s08_vox    start    93 s    -6.3 dB  stems 6
  s08_bass   start    93 s    -3.6 dB  stems 6
  s09_vox    start   120 s    -4.1 dB  stems 6
  s09_bass   start   120 s    -4.7 dB  stems 6
  s10_vox    start   131 s    -1.4 dB  stems 4
  s10_bass   start   131 s    -7.0 dB  stems 4

20 cases (10 vocals, 10 bass) as pid 40015


In [ ]:
"""V1 results: vocals and bass at n=10 each, shipped htdemucs_6s against htdemucs_ft."""

from statistics import mean, median

v1 = json.loads(Path("/content/v1.json").read_text())
for stem in ("vocals", "bass"):
    rows = [r for r in v1 if r["stem"] == stem]
    print(f"{stem}, n={len(rows)}")
    print(f"  {'case':10s} {'6s (shipped)':>13s} {'ft':>7s} {'delta':>7s}")
    for row in rows:
        print(f"  {row['case']:10s} {row['current 6s']:13.3f} {row['ft']:7.3f} "
              f"{row['ft'] - row['current 6s']:+7.3f}")
    before = [r["current 6s"] for r in rows]
    after = [r["ft"] for r in rows]
    wins = sum(1 for b, a in zip(before, after) if a > b)
    print(f"  mean   {mean(before):.3f} -> {mean(after):.3f}  ({mean(after) - mean(before):+.3f})")
    print(f"  median {median(before):.3f} -> {median(after):.3f}")
    print(f"  ft better on {wins}/{len(rows)}; worst single loss "
          f"{min(a - b for b, a in zip(before, after)):+.3f}\n")


vocals, n=10
  case        6s (shipped)      ft   delta
  s01_vox            0.638   0.651  +0.013
  s02_vox            0.742   0.768  +0.025
  s03_vox            0.862   0.896  +0.034
  s04_vox            0.835   0.840  +0.004
  s05_vox            0.691   0.705  +0.014
  s06_vox            0.942   0.925  -0.017
  s07_vox            0.814   0.830  +0.016
  s08_vox            0.667   0.677  +0.011
  s09_vox            0.866   0.839  -0.028
  s10_vox            0.780   0.839  +0.059
  mean   0.784 -> 0.797  (+0.013)
  median 0.797 -> 0.834
  ft better on 8/10; worst single loss -0.028

bass, n=10
  case        6s (shipped)      ft   delta
  s01_bass           0.713   0.755  +0.041
  s02_bass           0.754   0.786  +0.032
  s03_bass           0.619   0.669  +0.050
  s04_bass           0.681   0.722  +0.041
  s05_bass           0.706   0.677  -0.029
  s06_bass           0.676   0.640  -0.036
  s07_bass           0.684   0.725  +0.041
  s08_bass           0.767   0.759  -0.008
  s09_bass 

In [ ]:
"""V2: settle the note-length constants for bass and vocals now that each has ten cases.

Production gives bass 127.70 ms (from a synthetic curve) and vocals 30 ms. Two real cases each had
hinted bass prefers 30 ms and vocals dislike 127.70 ms; ten each can decide it.
"""

V2 = Path("/content/s2_score.py").read_text()
V2 = V2.replace('GUITAR_MS, DEFAULT_MS = 127.7, 30.0', 'SETTINGS = {"30 ms": 30.0, "127.7 ms": 127.7}')
V2 = V2.replace('''        for case in spec:
            ms = GUITAR_MS if case["stem"] == "guitar" else DEFAULT_MS
            case_dir = Path(case["dir"])
            reference = mono(case_dir / "reference.wav")
            n = len(reference)
            truth = notes(reference, ms)
            current = mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{case['stem']}.wav", n)
            row = {"case": case["id"], "stem": case["stem"],
                   "current 6s": f1(truth, notes(current, ms))}
            for variant_dir in sorted((out_root / case["id"]).iterdir()):
                path = variant_dir / f"{case['stem']}.wav"
                if path.is_file():
                    row[variant_dir.name] = f1(truth, notes(mono(path, n), ms))
            rows.append(row)''', '''        for case in spec:
            case_dir = Path(case["dir"])
            reference = mono(case_dir / "reference.wav")
            n = len(reference)
            current = mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{case['stem']}.wav", n)
            row = {"case": case["id"], "stem": case["stem"]}
            for label, ms in SETTINGS.items():
                row[label] = f1(notes(reference, ms), notes(current, ms))
            rows.append(row)''')
V2 = V2.replace('"/content/s2.json"', '"/content/v2.json"').replace('"/content/v1.json"', '"/content/v2.json"')
Path("/content/v2_score.py").write_text(V2)

done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/v2_score.py", "/content/v1_spec.json", "/content/v1"],
    env=gate_env, capture_output=True, text=True, timeout=3600,
)
if done.returncode:
    print(done.stderr[-2500:])
else:
    v2 = json.loads(Path("/content/v2.json").read_text())
    for stem, shipped in (("vocals", "30 ms"), ("bass", "127.7 ms")):
        rows = [r for r in v2 if r["stem"] == stem]
        short = [r["30 ms"] for r in rows]
        long_ = [r["127.7 ms"] for r in rows]
        wins = sum(1 for s, l in zip(short, long_) if s > l)
        print(f"{stem:7s} n={len(rows)}  30 ms {mean(short):.3f}   127.7 ms {mean(long_):.3f}   "
              f"30 ms better on {wins}/{len(rows)}   production uses {shipped}")


vocals  n=10  30 ms 0.784   127.7 ms 0.756   30 ms better on 8/10   production uses 30 ms
bass    n=10  30 ms 0.699   127.7 ms 0.687   30 ms better on 6/10   production uses 127.7 ms


In [ ]:
"""Rebuild the shipped-quality table with the larger samples, each class at its shipped setting."""

from statistics import mean

v2 = json.loads(Path("/content/v2.json").read_text())
recheck_by_case = {r["case"]: r for r in recheck}
shipped = {
    # vocals ship at 30 ms, bass at 127.70 ms (via the stem vote), guitar at 127.70 ms.
    "vocals": [r["30 ms"] for r in v2 if r["stem"] == "vocals"]
              + [recheck_by_case[c][BASE_KEY]["demucs"] for c in ("mend_vox", "nice_guy_vox")],
    "bass": [r["127.7 ms"] for r in v2 if r["stem"] == "bass"]
            + [recheck_by_case[c][NEW_KEY]["demucs"] for c in ("ex_bass", "evening_bass")],
    "guitar": [r[NEW_KEY]["demucs"] for r in recheck
               if r["requested_stem"] == "guitar" and r["case"] not in INVALID],
    "piano": [r[BASE_KEY]["demucs"] for r in recheck if r["requested_stem"] == "piano"],
}
other_rows = json.loads(Path("/content/other_routing.json").read_text())
shipped["other"] = [r["other"] for r in other_rows]

print(f"  {'class':8s} {'n':>3s} {'notes':>7s}   evidence")
print("  " + "-" * 64)
evidence = {"vocals": "10 songs plus 2", "bass": "10 songs plus 2",
            "guitar": "8 mixtures plus 360 labelled GuitarSet excerpts",
            "piano": "3 mixtures, thin", "other": "4 mixtures, thin"}
for name, values in shipped.items():
    print(f"  {name:8s} {len(values):3d} {mean(values) * 100:6.1f}%   {evidence[name]}")
reachable = shipped["vocals"] + shipped["bass"] + shipped["guitar"] + shipped["piano"]
print("  " + "-" * 64)
print(f"  {'named 4':8s} {len(reachable):3d} {mean(reachable) * 100:6.1f}%   vocals, bass, guitar, piano")

size = archive_path.stat().st_size / 2**30 if archive_path.exists() else 0.0
print(f"  archive {size:.2f} of 10.34 GiB")


  class      n   notes   evidence
  ----------------------------------------------------------------
  vocals    12   76.7%   10 songs plus 2
  bass      12   69.4%   10 songs plus 2
  guitar     8   67.1%   8 mixtures plus 360 labelled GuitarSet excerpts
  piano      3   18.4%   3 mixtures, thin
  other      4   13.4%   4 mixtures, thin
  ----------------------------------------------------------------
  named 4   35   67.0%   vocals, bass, guitar, piano

  CHAIN_START
  
  archive 9.02 of 10.34 GiB


## The final test: eight draws, ranked, on the holdout


In [ ]:
"""F2, the final test: Meta's documented configuration on the untouched holdout.

Eight sequential draws with span prediction, all kept on Drive, ranked afterwards by CLAP, Judge and
the paper's 5 CLAP + 1 Judge, on the same pool. Rule: the paper's ranker within 0.03 N1 of Demucs on
the holdout makes SAM the music default. Non-music runs with spans and one draw. Oracle is reported
as unavailable at runtime.
"""

F2_OUT = F_OUT / "holdout"
F2_OUT.mkdir(parents=True, exist_ok=True)
f2_music = []
for case in json.loads((BASE / "runs" / "holdout-004" / "config.json").read_text())["cases"]:
    f2_music.append({"id": case["id"], "dir": str(BASE / "runs" / "holdout-004" / case["id"]),
                     "prompt": case["prompt"], "requested": requested_stem(case["prompt"])})
f2_events = [{"id": c["id"], "dir": str(BASE / "runs" / "nonmusic-008" / c["id"]), "prompt": c["prompt"], "absent": c["absent"]}
             for c in json.loads((BASE / "runs" / "nonmusic-008" / "manifest.json").read_text())]
Path("/content/f2_spec.json").write_text(json.dumps({"music": f2_music, "events": f2_events}))

F2_SAM = r'''
import gc, json, sys
from pathlib import Path
import librosa, numpy as np, soundfile as sf

RATE, DRAWS = 48000, 8
SPAN_ONLY = {"visual_ranker": None, "text_ranker": None}


def mix_of(folder):
    audio, rate = sf.read(str(folder / "mix.wav"), dtype="float64", always_2d=True)
    x = audio.mean(axis=1)
    return librosa.resample(x, orig_sr=rate, target_sr=RATE) if rate != RATE else x


def main():
    import torch
    from huggingface_hub import snapshot_download
    from sam_audio import SAMAudio, SAMAudioProcessor
    from sam_audio.model.config import ClapRankerConfig, JudgeRankerConfig
    from sam_audio.ranking import create_ranker

    spec, out_root, revision = json.loads(Path(sys.argv[1]).read_text()), Path(sys.argv[2]), sys.argv[3]
    snapshot = snapshot_download("facebook/sam-audio-large", revision=revision,
                                 allow_patterns=["*.json", "*.pt", "*.safetensors"])
    model = SAMAudio.from_pretrained(snapshot, **SPAN_ONLY).eval().cuda()
    processor = SAMAudioProcessor.from_pretrained(snapshot)

    def draw(audio, prompt, seed):
        batch = processor(audios=[torch.tensor(audio, dtype=torch.float32).unsqueeze(0)], descriptions=[prompt]).to("cuda")
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        with torch.inference_mode():
            result = model.separate(batch, predict_spans=True, reranking_candidates=1)
        return result.target[0].detach().float().cpu().numpy().reshape(-1)[: len(audio)]

    for case in spec["events"]:
        destination = out_root / case["id"]
        destination.mkdir(parents=True, exist_ok=True)
        path = destination / "spans.wav"
        if not path.is_file():
            sf.write(str(path), draw(mix_of(Path(case["dir"])), case["prompt"], 1000), RATE, subtype="FLOAT")
        print(f"  event {case['id']}", flush=True)
    for case in spec["music"]:
        destination = out_root / case["id"]
        destination.mkdir(parents=True, exist_ok=True)
        mix = mix_of(Path(case["dir"]))
        for index in range(DRAWS):
            path = destination / f"candidate-{index}.wav"
            if not path.is_file():
                sf.write(str(path), draw(mix, case["prompt"], 1000 + index), RATE, subtype="FLOAT")
        print(f"  music {case['id']}", flush=True)
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

    rankers = {"clap": create_ranker(ClapRankerConfig()).eval().cuda(),
               "judge": create_ranker(JudgeRankerConfig()).eval().cuda()}
    for case in spec["music"]:
        destination = out_root / case["id"]
        mix = mix_of(Path(case["dir"]))
        drawn = [sf.read(str(destination / f"candidate-{i}.wav"), dtype="float32")[0] for i in range(DRAWS)]
        shared = min(len(mix), *(len(x) for x in drawn))
        pool = torch.tensor(np.stack([x[:shared] for x in drawn]), device="cuda")
        source = torch.tensor(mix[:shared], dtype=torch.float32, device="cuda").unsqueeze(0).expand(DRAWS, -1)
        scores = {}
        torch.manual_seed(1000)
        for name, ranker in rankers.items():
            with torch.inference_mode():
                values = ranker(input_audio=[source], extracted_audio=[pool], descriptions=[case["prompt"]], sample_rate=RATE)
            scores[name] = [float(v) for v in values[0].detach().float().cpu().tolist()]
        (destination / "scores.json").write_text(json.dumps(scores))
        print(f"  ranked {case['id']}", flush=True)


if __name__ == "__main__":
    main()
'''
F2_SCORE = r'''
import json, sys, tempfile
from pathlib import Path
import librosa, mir_eval, numpy as np, soundfile as sf

RATE, DRAWS = 48000, 8
STEMS = ("vocals", "drums", "bass", "guitar", "piano", "other")


def mono(path, n=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    x = audio.mean(axis=1)
    if rate != RATE:
        x = librosa.resample(x, orig_sr=rate, target_sr=RATE)
    return x if n is None else x[:n]


def si_sdr(ref, est):
    n = min(len(ref), len(est))
    t, a = ref[:n] - ref[:n].mean(), est[:n] - est[:n].mean()
    p = float(t @ t)
    if p < 1e-12:
        return float("nan")
    proj = t * float(a @ t) / p
    err = a - proj
    return float(10 * np.log10((proj @ proj + 1e-12) / (err @ err + 1e-12)))


def hz(v):
    return 440.0 * 2 ** ((np.asarray(v, dtype=float) - 69) / 12)


def f1(truth, events):
    if not truth or not events:
        return 0.0
    r = np.array([[e[0], e[1]] for e in truth]).reshape(-1, 2)
    e = np.array([[x[0], x[1]] for x in events]).reshape(-1, 2)
    return float(mir_eval.transcription.precision_recall_f1_overlap(
        r, hz([x[2] for x in truth]), e, hz([x[2] for x in events]),
        onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None)[2])


def main():
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import Model, predict

    spec, out_root = json.loads(Path(sys.argv[1]).read_text()), Path(sys.argv[2])
    model = Model(ICASSP_2022_MODEL_PATH)
    result = {"music": [], "events": []}
    for case in spec["events"]:
        folder, spans = Path(case["dir"]), mono(out_root / case["id"] / "spans.wav")
        mix = mono(folder / "mix.wav")
        row = {"id": case["id"], "prompt": case["prompt"], "absent": case["absent"],
               "rms_to_mix_db": float(20 * np.log10(max(np.sqrt(np.mean(spans ** 2)), 1e-12) / np.sqrt(np.mean(mix ** 2))))}
        if not case["absent"]:
            reference = mono(folder / "reference.wav")
            row["sdri"] = si_sdr(reference, spans) - si_sdr(reference, mix)
        result["events"].append(row)
    with tempfile.TemporaryDirectory() as tmp:
        staging = Path(tmp) / "clip.wav"

        def notes(x):
            peak = float(np.max(np.abs(x)))
            sf.write(str(staging), (x * (0.891 / peak) if peak > 0.01 else x).astype(np.float32), RATE, subtype="FLOAT")
            return [(e[0], e[1], e[2]) for e in predict(staging, model, onset_threshold=0.5, frame_threshold=0.3,
                                                        minimum_note_length=30.0, melodia_trick=True)[2]]

        for case in spec["music"]:
            folder, out = Path(case["dir"]), out_root / case["id"]
            reference = mono(folder / "reference.wav")
            n = len(reference)
            truth = notes(reference)
            stems = {s: mono(folder / "demucs" / "htdemucs_6s" / "mix" / f"{s}.wav", n) for s in STEMS}
            pool = [mono(out / f"candidate-{i}.wav", n) for i in range(DRAWS)]
            n1 = [f1(truth, notes(x)) for x in pool]
            closest = [max(STEMS, key=lambda s: si_sdr(stems[s], x)) for x in pool]
            scores = json.loads((out / "scores.json").read_text())
            paper = [5 * c + j for c, j in zip(scores["clap"], scores["judge"])]
            pick = {"first": 0, "clap": int(np.argmax(scores["clap"])), "judge": int(np.argmax(scores["judge"])),
                    "paper": int(np.argmax(paper)), "oracle": int(np.argmax(n1))}
            result["music"].append({"id": case["id"], "prompt": case["prompt"], "requested": case["requested"],
                                    "demucs_n1": f1(truth, notes(stems[case["requested"]])),
                                    "n1": n1, "closest": closest, "pick": pick})
            print(f"  scored {case['id']}", flush=True)
    Path("/content/f2.json").write_text(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()
'''
Path("/content/f2_sam.py").write_text(F2_SAM)
Path("/content/f2_score.py").write_text(F2_SCORE)
lines = ["echo F2_START",
         f"{UV} run --no-sync --project {PROJECTS['sam']} python /content/f2_sam.py /content/f2_spec.json {F2_OUT} {REVISION} || exit 1",
         f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['pitch']} python /content/f2_score.py /content/f2_spec.json {F2_OUT} || exit 1",
         "echo F2_DONE"]
(WORK / "f2.sh").write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/f2.log", "/content/f2.pid", "F2_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(WORK / "f2.sh")], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"final test: {len(f2_music)} music, {len(f2_events)} events, pid {job.pid}")


final test: 10 music, pid 12655


In [ ]:
"""F2 results against the frozen rule, recorded on Drive. Waits at most 6 minutes, starts nothing."""

import json
import time
from pathlib import Path
from statistics import mean, median

deadline = time.monotonic() + 360
while time.monotonic() < deadline and "F2_DONE" not in Path("/content/f2.log").read_text(errors="replace"):
    time.sleep(5)
if not Path("/content/f2.json").exists():
    print("not finished:\n", Path("/content/f2.log").read_text(errors="replace")[-600:])
else:
    data = json.loads(Path("/content/f2.json").read_text())
    music = data["music"]
    ways = ("first", "clap", "judge", "paper", "oracle")
    print(f"  {'case':18s} {'prompt':22s} {'demucs':>7s}" + "".join(f"{w:>8s}" for w in ways) + "  paper resembles")
    for r in music:
        print(f"  {r['id']:18s} {r['prompt']:22s} {r['demucs_n1']:7.3f}" + "".join(f"{r['n1'][r['pick'][w]]:8.3f}" for w in ways)
              + f"  {r['closest'][r['pick']['paper']]}")
    covered = [r for r in music if r["id"] not in {"evening_organ", "summer_synth", "mend_cello"}]
    for title, group in (("all ten", music), ("five named categories only", covered)):
        print(f"\n{title}, n={len(group)}: Demucs {mean(r['demucs_n1'] for r in group):.3f}")
        for w in ways:
            value = mean(r["n1"][r["pick"][w]] for r in group)
            wrong = sum(1 for r in group if r["closest"][r["pick"][w]] != r["requested"])
            label = " (runtime unavailable)" if w == "oracle" else ""
            print(f"  {w:7s} N1 {value:.3f}  gap to Demucs {value - mean(r['demucs_n1'] for r in group):+.3f}  wrong source {wrong}/{len(group)}{label}")
    paper = mean(r["n1"][r["pick"]["paper"]] for r in music)
    demucs = mean(r["demucs_n1"] for r in music)
    print(f"\nrule: paper's ranker within 0.03 of Demucs -> {paper - demucs:+.3f}: {'MET, SAM default' if paper >= demucs - 0.03 else 'NOT MET, Demucs stays'}")


  case               prompt                  demucs   first    clap   judge   paper  oracle  paper resembles
  nice_guy_vox       lead vocals              0.659   0.806   0.729   0.782   0.729   0.806  vocals
  nice_guy_solo      lead electric guitar     0.847   0.219   0.540   0.797   0.797   0.797  guitar
  karma_piano        piano                    0.460   0.124   0.100   0.114   0.100   0.133  bass
  karma_acoustic     acoustic guitar          0.743   0.090   0.751   0.751   0.751   0.751  guitar
  evening_organ      organ                    0.268   0.013   0.000   0.000   0.000   0.045  guitar
  evening_bass       bass guitar              0.732   0.766   0.713   0.795   0.795   0.795  bass
  summer_acoustic    acoustic guitar          0.817   0.793   0.787   0.811   0.787   0.819  guitar
  summer_synth       synthesizer              0.135   0.000   0.000   0.000   0.000   0.008  bass
  mend_cello         cello                    0.101   0.000   0.000   0.000   0.000   0.015  voca